In [ ]:
# !pip install optuna

In [ ]:

DATA_PATH   = "your_data.csv"      
TARGET_COL  = "value"              
SEQ_SIZE    = 24                  
NUM_STEP    = 1                    
TRAIN_RATIO = 0.8                
VAL_RATIO   = 0.1               
SEED        = 42
N_TRIALS    = 100              
OUTPUT_DIR  = "."             

In [ ]:
import math, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn import metrics
from sklearn.preprocessing import MinMaxScaler
import optuna

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

In [ ]:

df = pd.read_csv(DATA_PATH)[TARGET_COL]

train_size = int(len(df) * TRAIN_RATIO)
val_size   = int(len(df) * VAL_RATIO)

train = df.iloc[:train_size]
val   = df.iloc[train_size : train_size + val_size]
test  = df.iloc[train_size + val_size:]

scaler = MinMaxScaler(feature_range=(0, 1))
train = scaler.fit_transform(train.values.reshape(-1, 1))
val   = scaler.transform(val.values.reshape(-1, 1))
test  = scaler.transform(test.values.reshape(-1, 1))

print(f"Train size: {len(train)} | Val size: {len(val)} | Test size: {len(test)}")

In [ ]:

def to_sequences(seq_size, obs, num_step):
    x, y = [], []
    for i in range(len(obs) - seq_size - num_step + 1):
        x.append(obs[i : i + seq_size])
        y.append(obs[i + seq_size : i + seq_size + num_step])
    return (torch.tensor(np.array(x), dtype=torch.float32).view(-1, seq_size, 1),
            torch.tensor(np.array(y), dtype=torch.float32).view(-1, num_step))

x_train, y_train = to_sequences(SEQ_SIZE, train, NUM_STEP)
x_val,   y_val   = to_sequences(SEQ_SIZE, val,   NUM_STEP)
x_test,  y_test  = to_sequences(SEQ_SIZE, test,  NUM_STEP)

print(f"x_train: {x_train.shape} | x_val: {x_val.shape} | x_test: {x_test.shape}")

In [ ]:

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)


class TransformerLSTM(nn.Module):
    def __init__(self, input_dim=1, d_model=64, nhead=4, num_layers=2,
                 num_steps=1, hidden_size=64, dropout=0.2):
        super().__init__()
        self.encoder = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        encoder_layers = nn.TransformerEncoderLayer(d_model, nhead, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.lstm = nn.LSTM(input_size=d_model, hidden_size=hidden_size, num_layers=2, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_steps)

    def forward(self, x):
        x = self.encoder(x)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x, _ = self.lstm(x)
        return self.fc(x[:, -1, :])

In [ ]:
def objective(trial):
    d_model       = trial.suggest_categorical('d_model', [32, 64, 128, 256])
    nhead         = trial.suggest_categorical('nhead', [2, 4, 8])
    num_layers    = trial.suggest_int('num_layers', 1, 3)
    hidden_size   = trial.suggest_categorical('hidden_size', [16, 32, 64, 128])
    dropout       = trial.suggest_float('dropout', 0.1, 0.5)
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True)
    batch_size    = trial.suggest_categorical('batch_size', [16, 32, 64, 128])

    model = TransformerLSTM(input_dim=1, d_model=d_model, nhead=nhead,
                            num_layers=num_layers, num_steps=NUM_STEP,
                            hidden_size=hidden_size, dropout=dropout).to(device)

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=8)

    train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=False)
    val_loader   = DataLoader(TensorDataset(x_val, y_val),     batch_size=batch_size, shuffle=False)

    epochs = 150
    early_stop_count = 0
    min_val_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                val_losses.append(criterion(model(x_batch), y_batch).item())

        val_loss = np.mean(val_losses)
        scheduler.step(val_loss)

        if val_loss < min_val_loss:
            min_val_loss = val_loss
            early_stop_count = 0
        else:
            early_stop_count += 1
        if early_stop_count >= 8:
            break

    return min_val_loss


study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS)

best_params = study.best_params
print("Best trial:")
print(f"  Value (MSE): {study.best_value}")
for key, value in best_params.items():
    print(f"    {key}: {value}")

In [ ]:
set_seed(SEED)
BATCH_SIZE = best_params['batch_size']

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=BATCH_SIZE, shuffle=False)
val_loader   = DataLoader(TensorDataset(x_val, y_val),     batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(TensorDataset(x_test, y_test),   batch_size=BATCH_SIZE, shuffle=False)

model = TransformerLSTM(
    input_dim=1,
    d_model=best_params['d_model'],
    nhead=best_params['nhead'],
    num_layers=best_params['num_layers'],
    num_steps=NUM_STEP,
    hidden_size=best_params['hidden_size'],
    dropout=best_params['dropout'],
).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=best_params['learning_rate'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=5)

epochs = 1000
early_stop_count = 0
min_val_loss = float('inf')
train_start_time = time.time()

for epoch in range(epochs):
    model.train()
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x_batch), y_batch)
        loss.backward()
        optimizer.step()

  
    model.eval()
    val_losses = []
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            val_losses.append(criterion(model(x_batch), y_batch).item())

    val_loss = np.mean(val_losses)
    scheduler.step(val_loss)

    if val_loss < min_val_loss:
        min_val_loss = val_loss
        early_stop_count = 0
    else:
        early_stop_count += 1

    if early_stop_count >= 8:
        print("Early stopping!")
        break

    print(f"Epoch {epoch + 1}/{epochs}, Val Loss: {val_loss:.5f}")

train_duration = time.time() - train_start_time
print(f"\nThoi gian huan luyen: {train_duration:.2f} giay")

In [ ]:
test_start_time = time.time()

model.eval()
predictions, ground_truth = [], []
with torch.no_grad():
    for x_batch, y_batch in test_loader:
        outputs = model(x_batch.to(device))
        predictions.append(outputs.cpu().numpy())
        ground_truth.append(y_batch.numpy())

test_duration = time.time() - test_start_time
print(f"Thoi gian kiem tra (Test Time): {test_duration:.4f} giay")

predictions = np.concatenate(predictions, axis=0)
y_test_arr  = np.concatenate(ground_truth, axis=0)

predictions_rescaled = scaler.inverse_transform(predictions)
y_test_rescaled      = scaler.inverse_transform(y_test_arr)


def evaluation_metric(actual, predicted, test_time=None):
    actual = np.array(actual)
    predicted = np.array(predicted)

    mse = metrics.mean_squared_error(actual, predicted)
    mae = metrics.mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mse)
    nrmse = rmse / (np.max(actual) - np.min(actual))
    r2 = metrics.r2_score(actual, predicted)

    epsilon = 1e-10 
    mape = np.mean(np.abs((actual - predicted) / (actual + epsilon))) * 100
    smape = np.mean(2 * np.abs(predicted - actual) /
                    (np.abs(actual) + np.abs(predicted) + epsilon)) * 100

    print(f"MSE        : {mse:.5f}")
    print(f"MAE        : {mae:.5f}")
    print(f"RMSE       : {rmse:.5f}")
    print(f"NRMSE      : {nrmse:.5f}")
    print(f"R2         : {r2:.5f}")
    print(f"MAPE       : {mape:.5f}%")
    print(f"SMAPE      : {smape:.5f}%")
    if test_time is not None:
        print(f"Test Time  : {test_time:.5f} s")


for step in range(predictions_rescaled.shape[1]):
    print(f"\n===== Sai so tai buoc {step+1} =====")
    evaluation_metric(y_test_rescaled[:, step], predictions_rescaled[:, step], test_time=test_duration)

print("\n===== Sai so tong hop tren tat ca cac buoc =====")
evaluation_metric(y_test_rescaled.flatten(), predictions_rescaled.flatten(), test_time=test_duration)

In [ ]:

step_cols = [str(i + 1) for i in range(predictions_rescaled.shape[1])]
df_all = pd.concat([
    pd.DataFrame(predictions_rescaled, columns=[f"Du bao buoc {c}" for c in step_cols]),
    pd.DataFrame(y_test_rescaled,      columns=[f"Thuc te buoc {c}" for c in step_cols]),
], axis=1)

out_path = f"{OUTPUT_DIR}/Transformer-LSTM_{NUM_STEP}step.csv"
df_all.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"Da luu: {out_path}")

In [ ]:

model.eval()
val_predictions_ec, val_ground_truth_ec = [], []

with torch.no_grad():
    for x_batch, y_batch in val_loader:
        outputs = model(x_batch.to(device))
        val_predictions_ec.append(outputs.cpu().numpy())
        val_ground_truth_ec.append(y_batch.numpy())

val_predictions_ec  = np.concatenate(val_predictions_ec,  axis=0)
val_ground_truth_ec = np.concatenate(val_ground_truth_ec, axis=0)

val_errors = val_ground_truth_ec - val_predictions_ec  

print(f"val_errors shape: {val_errors.shape}  (N_val, NUM_STEP)")
for s in range(NUM_STEP):
    print(f"  Cot {s+1:2d}: mean={val_errors[:, s].mean():.5f}  std={val_errors[:, s].std():.5f}")

In [ ]:

ERROR_LOOKBACK = 24  

def make_error_seq(error_1d, lookback):
    x, y = [], []
    for i in range(len(error_1d) - lookback):
        x.append(error_1d[i : i + lookback])
        y.append(error_1d[i + lookback])
    x = np.array(x, dtype=np.float32).reshape(-1, lookback, 1)
    y = np.array(y, dtype=np.float32).reshape(-1, 1)
    return torch.tensor(x), torch.tensor(y)

err_seq_per_step = [make_error_seq(val_errors[:, s], ERROR_LOOKBACK) for s in range(NUM_STEP)]

print(f"ERROR_LOOKBACK: {ERROR_LOOKBACK} | So cot sai so: {NUM_STEP}")
print(f"Moi cot -> x: {err_seq_per_step[0][0].shape}, y: {err_seq_per_step[0][1].shape}")

In [ ]:

class ErrorCorrectionLSTM(nn.Module):
    def __init__(self, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_size, num_layers=num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


def train_one_error_model(xs, ys, hidden_size=64, num_layers=2, dropout=0.2,
                          lr=1e-3, batch_size=32, max_epochs=300, patience=20):
    loader = DataLoader(TensorDataset(xs, ys), batch_size=batch_size, shuffle=False)

    m     = ErrorCorrectionLSTM(hidden_size, num_layers, dropout).to(device)
    opt   = optim.Adam(m.parameters(), lr=lr)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', factor=0.5, patience=10)
    crit  = nn.MSELoss()

    best_loss, best_state, no_improve = float('inf'), None, 0

    for epoch in range(max_epochs):
        m.train()
        losses = []
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(m(xb), yb)
            loss.backward()
            opt.step()
            losses.append(loss.item())
        avg = np.mean(losses)
        sched.step(avg)
        if avg < best_loss:
            best_loss  = avg
            best_state = {k: v.clone() for k, v in m.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            break

    m.load_state_dict(best_state)
    return m, best_loss


err_models = []
print(f"Train {NUM_STEP} mo hinh LSTM hieu chinh sai so...")
ec_train_start = time.time()

for s in range(NUM_STEP):
    xs, ys = err_seq_per_step[s]
    m, best_loss = train_one_error_model(xs, ys)
    err_models.append(m)
    print(f"  Cot {s+1:2d}/{NUM_STEP} | best MSE (scaled): {best_loss:.6f}")

print(f"\nTong thoi gian train error models: {time.time() - ec_train_start:.1f}s")

In [ ]:

def multi_step_inverse_transform(scaler, arr):
    out = np.zeros_like(arr)
    for c in range(arr.shape[1]):
        out[:, c] = scaler.inverse_transform(arr[:, c].reshape(-1, 1)).flatten()
    return out


model.eval()
raw_test_preds, raw_test_truth = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        raw_test_preds.append(model(xb.to(device)).cpu().numpy())
        raw_test_truth.append(yb.numpy())

raw_test_preds = np.concatenate(raw_test_preds, axis=0)
raw_test_truth = np.concatenate(raw_test_truth, axis=0)
test_errors_actual = raw_test_truth - raw_test_preds

predicted_corrections = np.zeros_like(raw_test_preds)

for s in range(NUM_STEP):
    err_models[s].eval()
    buf = val_errors[-ERROR_LOOKBACK:, s].copy()

    with torch.no_grad():
        for i in range(len(raw_test_preds)):
            x_in = torch.tensor(buf.reshape(1, ERROR_LOOKBACK, 1), dtype=torch.float32).to(device)
            predicted_corrections[i, s] = err_models[s](x_in).cpu().numpy()[0, 0]
            buf = np.append(buf[1:], test_errors_actual[i, s])  
    print(f"  Cot {s+1:2d}/{NUM_STEP} hieu chinh xong")

corrected_test_preds = raw_test_preds + predicted_corrections

raw_test_preds_rescaled       = multi_step_inverse_transform(scaler, raw_test_preds)
corrected_test_preds_rescaled = multi_step_inverse_transform(scaler, corrected_test_preds)
y_test_rescaled_ec            = multi_step_inverse_transform(scaler, raw_test_truth)

print(f"\nShape du bao cuoi: {corrected_test_preds_rescaled.shape}")

In [ ]:

for step in range(NUM_STEP):
    print(f"\n{'=' * 60}\nBUOC {step+1}/{NUM_STEP}\n{'=' * 60}")
    print("  >> TRUOC hieu chinh:")
    evaluation_metric(y_test_rescaled_ec[:, step], raw_test_preds_rescaled[:, step])
    print("  >> SAU hieu chinh:")
    evaluation_metric(y_test_rescaled_ec[:, step], corrected_test_preds_rescaled[:, step])

print(f"\n{'=' * 60}\nTONG HOP (flatten tat ca buoc)\n{'=' * 60}")
print("  >> TRUOC hieu chinh:")
evaluation_metric(y_test_rescaled_ec.flatten(), raw_test_preds_rescaled.flatten())
print("  >> SAU hieu chinh:")
evaluation_metric(y_test_rescaled_ec.flatten(), corrected_test_preds_rescaled.flatten())

In [ ]:

step_cols = [str(i + 1) for i in range(NUM_STEP)]

df_all_ec = pd.concat([
    pd.DataFrame(y_test_rescaled_ec,            columns=[f"Thuc te buoc {c}" for c in step_cols]),
    pd.DataFrame(raw_test_preds_rescaled,       columns=[f"Du bao goc buoc {c}" for c in step_cols]),
    pd.DataFrame(corrected_test_preds_rescaled, columns=[f"Du bao hieu chinh buoc {c}" for c in step_cols]),
], axis=1)

out_path_ec = f"{OUTPUT_DIR}/Transformer-LSTM-ErrorCorrection_{NUM_STEP}step.csv"
df_all_ec.to_csv(out_path_ec, index=False, encoding='utf-8-sig')
print(f"Da luu: {out_path_ec}")
print(df_all_ec.head())